In [17]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
data = pd.read_csv("patients.csv")
data.head()

,id,nom,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,medecin_id
0,1,Ahmed Baccouche,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0,15
1,2,Ghada Karray,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1,5
2,3,Hichem Ferchichi,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0,4
3,4,Amel Triki,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1,9
4,5,Omar Trabelsi,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0,7


In [19]:
print(f"Nombres des obersvations : {data.shape[0]}")
print(f"Nombre des variables : {data.shape[1]}")

Nombres des obersvations : 918
Nombre des variables : 15


In [20]:
print(f"Les informations générales sur le dataset : {data.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              918 non-null    int64  
 1   nom             918 non-null    object 
 2   Age             918 non-null    int64  
 3   Sex             918 non-null    object 
 4   ChestPainType   918 non-null    object 
 5   RestingBP       918 non-null    int64  
 6   Cholesterol     918 non-null    int64  
 7   FastingBS       918 non-null    int64  
 8   RestingECG      918 non-null    object 
 9   MaxHR           918 non-null    int64  
 10  ExerciseAngina  918 non-null    object 
 11  Oldpeak         918 non-null    float64
 12  ST_Slope        918 non-null    object 
 13  HeartDisease    918 non-null    int64  
 14  medecin_id      918 non-null    int64  
dtypes: float64(1), int64(8), object(6)
memory usage: 107.7+ KB
Les informations générales sur le dataset : None


In [21]:
#Informations sur les variables numériques 
data.describe()

,id,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,medecin_id
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,459.500000,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377,7.927015
std,265.148072,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414,4.322611
min,1.000000,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000,1.000000
25%,230.250000,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000,4.000000
50%,459.500000,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000,8.000000
75%,688.750000,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000,12.000000
max,918.000000,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000,15.000000


In [22]:
#La distribution des anomalies cardiaques dans le dataset
data["HeartDisease"].value_counts()

HeartDisease
1    508
0    410
Name: count, dtype: int64

### Repartition du variable prédiction et variable à prédire

In [23]:
# X = les informations pour prédire
X = data.drop(columns=["id", "nom", "medecin_id", "HeartDisease"])

# y = la réponse à prédire (0 = sain, 1 = maladie cardiaque)
y = data["HeartDisease"]

print("Forme de X :", X.shape)
print("Forme de y :", y.shape)
X.head()

Forme de X : (918, 11)
Forme de y : (918,)


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up


In [24]:
X[["Cholesterol", "RestingBP"]] = X[["Cholesterol", "RestingBP"]].replace(0, np.nan)
print(X[["Cholesterol", "RestingBP"]].isna().sum())

Cholesterol    172
RestingBP        1
dtype: int64


### Split the data

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test = X_test.copy()

print("Entraînement :", X_train.shape)
print("Test         :", X_test.shape)

print(X_train[["Cholesterol", "RestingBP"]].isna().sum())
print(X_test[["Cholesterol", "RestingBP"]].isna().sum())

Entraînement : (734, 11)
Test         : (184, 11)
Cholesterol    129
RestingBP        0
dtype: int64
Cholesterol    43
RestingBP       1
dtype: int64


### Remplacer les faux zéros avec le Médiane

In [26]:
from sklearn.impute import SimpleImputer

cols_imp = ["Cholesterol", "RestingBP"]
imputer_num = SimpleImputer(strategy="median")

# fit = calculer la médiane sur le TRAIN seulement
X_train[cols_imp] = imputer_num.fit_transform(X_train[cols_imp])

# transform = appliquer la médiane du train au TEST
X_test[cols_imp] = imputer_num.transform(X_test[cols_imp])

print(X_train[cols_imp].isna().sum())
print(X_test[cols_imp].isna().sum())
print("Médianes apprises :", imputer_num.statistics_)

Cholesterol    0
RestingBP      0
dtype: int64
Cholesterol    0
RestingBP      0
dtype: int64
Médianes apprises : [238. 130.]


### Encode the categorical variables

In [27]:
from sklearn.preprocessing import OneHotEncoder

# Les 5 colonnes qui contiennent du texte
cat_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]

ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")

# On apprend sur l'entraînement seulement, puis on applique au test
X_train_ohe = ohe.fit_transform(X_train[cat_cols])
X_test_ohe = ohe.transform(X_test[cat_cols])

print("Entraînement :", X_train_ohe.shape)
print("Test         :", X_test_ohe.shape)
print(ohe.get_feature_names_out(cat_cols))

Entraînement : (734, 9)
Test         : (184, 9)
['Sex_M' 'ChestPainType_ATA' 'ChestPainType_NAP' 'ChestPainType_TA'
 'RestingECG_Normal' 'RestingECG_ST' 'ExerciseAngina_Y' 'ST_Slope_Flat'
 'ST_Slope_Up']


### Join The numerical variables and the encoded columns

In [28]:
num_cols = ["Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak"]

X_train_final = np.hstack([X_train[num_cols].values, X_train_ohe])
X_test_final = np.hstack([X_test[num_cols].values, X_test_ohe])

print("Entraînement :", X_train_final.shape)
print("Test         :", X_test_final.shape)

Entraînement : (734, 15)
Test         : (184, 15)


### Train and Evaluate

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_final, y_train)

y_pred = model.predict(X_test_final)

print("Précision :", accuracy_score(y_test, y_pred))
print()
print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Précision : 0.875

Matrice de confusion :
[[71 11]
 [12 90]]

              precision    recall  f1-score   support

           0       0.86      0.87      0.86        82
           1       0.89      0.88      0.89       102

    accuracy                           0.88       184
   macro avg       0.87      0.87      0.87       184
weighted avg       0.88      0.88      0.88       184



In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

modeles = {
    "Régression logistique": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

for nom, m in modeles.items():
    scores = cross_val_score(m, X_train_final, y_train, cv=5, scoring="recall")
    print(f"{nom:22s} rappel moyen = {scores.mean():.3f} (± {scores.std():.3f})")

Régression logistique  rappel moyen = 0.869 (± 0.030)
Random Forest          rappel moyen = 0.899 (± 0.038)
Gradient Boosting      rappel moyen = 0.864 (± 0.032)


### Save the model 

In [30]:
import joblib

joblib.dump({
    "model": model,
    "ohe": ohe,
    "imputer": imputer_num,
    "num_cols": num_cols,
    "cat_cols": cat_cols
}, "model.pkl")

print("model.pkl sauvegardé")

model.pkl sauvegardé
